## 核心实现文件位置

这一机制并非集中在一个单一文件中，而是分布在 megatron/core/tensor_parallel/ 目录下。最关键的实现逻辑位于：
- https://github.com/NVIDIA/Megatron-LM/tree/core_v0.17.0/megatron/core/tensor_parallel
### megatron/core/tensor_parallel/layers.py：
  - 这是 TP 层的核心定义文件。其中定义了 ColumnParallelLinear、RowParallelLinear 等关键类。
  - 在这些类的 forward 方法中，<font color='red'>实现了计算（矩阵乘法）与通信（All-Gather / Reduce-Scatter）的交错逻辑。</font>
  - 代码中会显式地创建或使用非默认的 CUDA Stream（即“通信流”），将 NCCL 通信操作放入该流中，而将 GEMM（矩阵乘）操作保留在主计算流中，从而实现异步重叠。
  
### megatron/core/tensor_parallel/mappings.py：
  - 包含了具体的通信原语封装，
  - 如 gather_from_tensor_model_parallel_region 和 reduce_scatter_to_tensor_model_parallel_region。
  - 这些函数内部处理了 Stream 的切换和同步原语。
  
### megatron/core/utils.py 或专门的 stream 管理模块：
  - 通常包含获取或创建特定 Stream 的工具函数（例如 get_cuda_stream），用于管理主计算流和通信流的句柄。

## 代码层面的实现逻辑
在上述文件中，<font color='red'>双 CUDA 流调度</font>的具体代码逻辑通常表现为以下步骤：
1. <b>Stream 获取与切换</b>：
   - 在执行通信操作前，代码会通过类似 torch.cuda.current_stream() 获取当前默认的计算流，并切换到专门用于通信的非默认流。
2. <b>异步通信启动</b>：
   - 在通信流上发起 dist.all_gather 或 dist.reduce_scatter 操作。<font color='red'>由于是在不同流上，这些操作不会阻塞主计算流。</font>
3. <b>计算执行</b>：
   - 主计算流继续执行矩阵乘法（GEMM）。此时，GPU 的 Copy Engine（<font color='red'>负责通信数据传输</font>）和 Tensor Cores（<font color='red'>负责计算</font>）可以同时工作。
4. <b>同步等待</b>：
   - 在需要使用通信结果进行下一步计算之前，代码会插入同步点（如 stream.wait_event 或 torch.cuda.synchronize），确保数据已经传输完毕。

## 版本差异提示
- 需要注意的是，Megatron-LM 经历了多次重构。
  - 如果你使用的是较新的 NVIDIA Megatron-Core（独立出的核心库），上述路径是准确的。
  - 如果你使用的是较老版本的原始 Megatron-LM 仓库，相关代码可能位于 megatron/mpu/layers.py 或 megatron/model/transformer.py 中，但核心原理——利用多流实现计算通信重叠——是一致的。
